# GPR Preprocessing Pipeline

This notebook prepares groundwater telemetry data for Gaussian Process Regression (GPR). GPR operates on a fundamentally different paradigm from classical ML:

- **No lag features needed** — GPR's kernel models temporal correlation directly via the time index
- **Irregular sampling is native** — GPR handles non-uniform time steps without interpolation
- **Small data is fine** — GPR excels with hundreds to low thousands of points
- **O(n³) scaling** — covariance matrix inversion is cubic in the number of data points; this is the critical constraint

**Output:** Ready-to-use `X_train_scaled`, `X_test_scaled`, `y_train_scaled`, `y_test_scaled` arrays for Stage 2.

**Not included:** Model training, kernel design, prediction evaluation — those belong in Stage 2.

The preprocessing logic has been extracted into `ml/preprocessing/gpr.py` as reusable functions. This notebook demonstrates the pipeline step-by-step as a learning reference and can call those functions directly.

## 1. Imports

We import from `preprocessing.gpr` (the extracted function library) plus plotting utilities.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure ml/ is on the path so we can import our modules
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))

from preprocessing.gpr import (
    load_parquet,
    get_station_summary,
    build_time_index,
    detect_gaps,
    prepare_features,
    time_split,
    scale_features,
    save_scalers,
    full_pipeline,
    GWL_COL, TIME_COL,
)

## 2. Load the Cleaned Parquet

In [ ]:
PARQUET_PATH = '../data/processed/common.parquet'
df = load_parquet(PARQUET_PATH)
print(f'Loaded {len(df):,} rows × {len(df.columns)} columns')
df.head(3)

## 3. Station-Level Overview

GPR is applied **per station** — each well has its own time series. We need to understand the data distribution across stations before picking a prototype.

In [ ]:
station_counts = get_station_summary(df)
print(f'Total stations: {len(station_counts)}')
print(f'Points per station — min: {station_counts.min()}, max: {station_counts.max()}, median: {station_counts.median():.0f}')
print()
print('Top 10 stations by point count:')
station_counts.head(10)

### Scaling Consideration

GPR computes and inverts an n×n covariance matrix, giving it **O(n³)** time complexity and **O(n²)** memory. Practical limits:

| Points | Feasibility |
|--------|------------|
| < 1,000 | Fast, no issues |
| 1,000–5,000 | Workable with optimized kernels, ~minutes |
| 5,000–10,000 | Expensive; consider sparse approximations |
| > 10,000 | Likely infeasible for exact GPR |

If our prototype station exceeds a few thousand points, we will need to flag this for **sparse/inducing-point methods** in Stage 2.

## 4. Prototype Station Selection

We pick the station with the longest, most continuous record for initial development.

In [ ]:
STATION = station_counts.index[0]
station_df = df[df['Station'] == STATION].copy()
station_df = station_df.sort_values(TIME_COL).reset_index(drop=True)

print(f'Prototype station: {STATION}')
print(f'Total points: {len(station_df):,}')
print(f'Date range: {station_df[TIME_COL].min()} → {station_df[TIME_COL].max()}')
print(f'Duration: {(station_df[TIME_COL].max() - station_df[TIME_COL].min()).days} days')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(station_df[TIME_COL], station_df[GWL_COL], linewidth=0.5, alpha=0.8)
ax.set_title(f'Raw GWL Time Series — {STATION}')
ax.set_ylabel('Water Level (m)')
ax.set_xlabel('Date')
plt.tight_layout()
plt.show()

## 5. Verify Chronological Continuity

GPR does **not** require regular time sampling — the kernel handles irregular spacing natively. However, we should still **identify gaps** because:

1. Large gaps may indicate sensor failures or data quality issues
2. The gap structure informs kernel choice (e.g., Matern vs. RBF)
3. Knowing the typical sampling interval helps set kernel length-scale priors

We will **NOT** interpolate missing values. GPR models uncertainty over gaps naturally — interpolation would artificially constrain it.

In [ ]:
gaps = detect_gaps(station_df)
print(f'Gaps detected (> 9.0h): {len(gaps)}')
if len(gaps) > 0:
    print('\nGap details (first 10):')
    print(gaps[[TIME_COL, 'gap_hours']].head(10).to_string(index=False))

In [ ]:
time_diffs = station_df[TIME_COL].diff().dropna()
hours = time_diffs.dt.total_seconds() / 3600

fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(hours, bins=50, edgecolor='black', linewidth=0.5)
ax.axvline(6, color='red', linestyle='--', label='Expected (6h)')
ax.axvline(9, color='orange', linestyle='--', label='Gap threshold (9h)')
ax.set_xlabel('Time step (hours)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of consecutive reading intervals')
ax.legend()
plt.show()

## 6. Convert Time to Numeric Index

GPR requires numeric input features. We convert `Data Acquisition Time` to **hours elapsed since the station's first reading**. This is the primary input feature (X).

Using elapsed hours (rather than raw timestamps) gives the kernel a meaningful distance metric: two readings 24 hours apart have a time-distance of 24, regardless of calendar dates.

In [ ]:
station_df = build_time_index(station_df)
print(f'Time index range: 0 → {station_df["time_hours"].max():.1f} hours')
print(f'That is {station_df["time_hours"].max() / 24:.1f} days')
station_df[[TIME_COL, 'time_hours', GWL_COL]].head()

### Why Not Lag Features?

Classical time series models (AR, ARIMA) require explicit lag features because they lack a built-in notion of temporal correlation. GPR's kernel function (e.g., RBF, Matern) **already models smoothness and correlation as a function of time distance**. Adding lag features would:

- Be redundant with the kernel's temporal modeling
- Potentially conflict with the kernel's assumptions
- Increase dimensionality unnecessarily

**Future experiment:** If the kernel-based approach underperforms, we could explore adding lag features or switching to a multi-output kernel that explicitly models autoregressive structure. But the kernel-first approach should be tried first.

## 7. Feature Engineering

### Primary Feature
- `time_hours` — elapsed hours since first reading (the core GPR input)

### Optional: Seasonal Encodings
If we later explore **periodic kernels** (e.g., `ExpSineSquared` in scikit-learn), we can add day-of-year or hour-of-day as additional features. For now, we keep it simple — the basic RBF/Matern kernel operates on `time_hours` alone.

In [ ]:
X_all, y_all = prepare_features(station_df)

print(f'Feature matrix shape: {X_all.shape}')
print(f'Target vector shape:  {y_all.shape}')
print(f'\nX range: [{X_all.min():.1f}, {X_all.max():.1f}] hours')
print(f'y range: [{y_all.min():.3f}, {y_all.max():.3f}] meters')

## 8. Time-Based Train/Test Split

For time series, **never shuffle** — we must preserve temporal order. The split is:
- **Train:** Earlier readings
- **Test:** Later readings

This simulates the real deployment scenario: we train on historical data and predict future water levels. Using an 80/20 split.

In [ ]:
split = time_split(X_all, y_all, station_df)

X_train, X_test = split['X_train'], split['X_test']
y_train, y_test = split['y_train'], split['y_test']

print(f'Train: {len(X_train):,} samples (80%)')
print(f'  Time range: {pd.Timestamp(split["split_time"]) → ...')
print(f'Test:  {len(X_test):,} samples (20%)')
print(f'\nNo temporal overlap between train and test.')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(station_df[TIME_COL].values[:len(X_train)], y_train, linewidth=0.5, alpha=0.8, label='Train', color='steelblue')
ax.plot(station_df[TIME_COL].values[len(X_train):], y_test, linewidth=0.5, alpha=0.8, label='Test', color='coral')
ax.axvline(pd.Timestamp(split['split_time']), color='black', linestyle='--', linewidth=1, alpha=0.7, label='Split boundary')
ax.set_title(f'Train/Test Split — {STATION}')
ax.set_ylabel('Water Level (m)')
ax.set_xlabel('Date')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Feature Scaling

**Critical rule:** Fit the scaler **only on training data**, then transform both train and test. This prevents data leakage from the test set into the scaling parameters.

We save the fitted scaler to disk so Stage 2 can apply the same transformation to new data at inference time.

**Note on GPR + scaling:** Standardizing the target (y) is particularly important for GPR because the kernel's noise parameter (`alpha`) and length-scale are sensitive to the scale of the target variable. An unscaled target can lead to poor kernel parameter optimization.

In [ ]:
scaled = scale_features(X_train, X_test, y_train, y_test)

X_train_scaled = scaled['X_train_scaled']
X_test_scaled = scaled['X_test_scaled']
y_train_scaled = scaled['y_train_scaled']
y_test_scaled = scaled['y_test_scaled']
x_scaler = scaled['x_scaler']
y_scaler = scaled['y_scaler']

print('Scaler statistics (fit on training data only):')
print(f'  X scaler — mean: {x_scaler.mean_[0]:.2f}, std: {x_scaler.scale_[0]:.2f}')
print(f'  y scaler — mean: {y_scaler.mean_[0]:.4f}, std: {y_scaler.scale_[0]:.4f}')
print()
print(f'X_train_scaled range: [{X_train_scaled.min():.2f}, {X_train_scaled.max():.2f}]')
print(f'X_test_scaled range:  [{X_test_scaled.min():.2f}, {X_test_scaled.max():.2f}]')
print(f'y_train_scaled range: [{y_train_scaled.min():.2f}, {y_train_scaled.max():.2f}]')
print(f'y_test_scaled range:  [{y_test_scaled.min():.2f}, {y_test_scaled.max():.2f}]')

In [ ]:
save_scalers(x_scaler, y_scaler)
print(f'Saved scalers to ../artifacts/')

## 10. Save Processed Dataset

Saving the GPR-ready arrays and metadata for the Streamlit visualization app and Stage 2.

In [ ]:
output_df = pd.DataFrame({
    'station': STATION,
    'time': station_df[TIME_COL].values[:len(X_all)],
    'time_hours': X_all.ravel(),
    'gwl': y_all,
    'gwl_scaled': np.concatenate([y_train_scaled, y_test_scaled]),
    'time_hours_scaled': np.concatenate([X_train_scaled.ravel(), X_test_scaled.ravel()]),
    'split': ['train'] * len(X_train) + ['test'] * len(X_test)
})

OUTPUT_PATH = '../data/processed/gpr_ready.parquet'
output_df.to_parquet(OUTPUT_PATH, index=False)
print(f'Saved GPR-ready dataset → {OUTPUT_PATH}')
print(f'  Rows: {len(output_df):,}')
print(f'  Columns: {list(output_df.columns)}')

## 11. Summary & Ready Variables

The following variables are ready for Stage 2 (GPR model training):

| Variable | Shape | Description |
|----------|-------|-------------|
| `X_train_scaled` | (n_train, 1) | Scaled training features (hours since start) |
| `X_test_scaled` | (n_test, 1) | Scaled test features |
| `y_train_scaled` | (n_train,) | Scaled training target (GWL) |
| `y_test_scaled` | (n_test,) | Scaled test target |
| `x_scaler` | StandardScaler | Fitted on X_train — reuse for new data |
| `y_scaler` | StandardScaler | Fitted on y_train — reuse for new data |
| `STATION` | str | Station identifier |

Saved artifacts:
- `../artifacts/x_scaler.pkl`
- `../artifacts/y_scaler.pkl`
- `../data/processed/gpr_ready.parquet`

**Next steps (Stage 2):** Kernel selection (see `models/gaussian_process.py`), `GaussianProcessRegressor` instantiation, hyperparameter optimization, prediction with uncertainty bands (see `training/train_gpr.py`).

In [ ]:
print('=== GPR Preprocessing Complete ===')
print(f'Station: {STATION}')
print(f'Total points: {len(X_all):,}')
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'Features: {X_all.shape[1]} (time_hours)')
print(f'\nReady for Stage 2: X_train_scaled, X_test_scaled, y_train_scaled, y_test_scaled')